In [16]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.v_pd_hf_tweezer_squeeze_power = 3.94
                                 
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 500.e-6
                
                self.p.amp_imaging = {img_amp}
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 1.21 * np.pi

                self.p.t_tweezer_hold = 15.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 15

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(ramp_down_painting=True,squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [17]:
eBuilder = ExptBuilder()

In [18]:
img_amp = np.linspace(.2,2.5,50)

freq_raman = np.array([1.19494357e+08, 1.19501388e+08, 1.19508419e+08, 1.19515449e+08,
       1.19522480e+08, 1.19529511e+08, 1.19536542e+08, 1.19543572e+08,
       1.19550603e+08, 1.19557634e+08, 1.19564665e+08, 1.19571695e+08,
       1.19578726e+08, 1.19585757e+08, 1.19592788e+08, 1.19599819e+08,
       1.19606849e+08, 1.19613880e+08, 1.19620911e+08, 1.19627942e+08,
       1.19634972e+08, 1.19642003e+08, 1.19649034e+08, 1.19656065e+08,
       1.19663095e+08, 1.19670126e+08, 1.19677157e+08, 1.19684188e+08,
       1.19691218e+08, 1.19698249e+08, 1.19705280e+08, 1.19712311e+08,
       1.19719341e+08, 1.19726372e+08, 1.19733403e+08, 1.19740434e+08,
       1.19747464e+08, 1.19754495e+08, 1.19761526e+08, 1.19768557e+08,
       1.19775587e+08, 1.19782618e+08, 1.19789649e+08, 1.19796680e+08,
       1.19803710e+08, 1.19810741e+08, 1.19817772e+08, 1.19824803e+08,
       1.19831834e+08, 1.19838864e+08])

# img_amp = np.linspace(.5,1.5,50)

# freq_raman = np.array([1.19539293e+08, 1.19542350e+08, 1.19545407e+08, 1.19548463e+08,
#        1.19551520e+08, 1.19554577e+08, 1.19557634e+08, 1.19560691e+08,
#        1.19563748e+08, 1.19566805e+08, 1.19569861e+08, 1.19572918e+08,
#        1.19575975e+08, 1.19579032e+08, 1.19582089e+08, 1.19585146e+08,
#        1.19588202e+08, 1.19591259e+08, 1.19594316e+08, 1.19597373e+08,
#        1.19600430e+08, 1.19603487e+08, 1.19606544e+08, 1.19609600e+08,
#        1.19612657e+08, 1.19615714e+08, 1.19618771e+08, 1.19621828e+08,
#        1.19624885e+08, 1.19627942e+08, 1.19630998e+08, 1.19634055e+08,
#        1.19637112e+08, 1.19640169e+08, 1.19643226e+08, 1.19646283e+08,
#        1.19649340e+08, 1.19652396e+08, 1.19655453e+08, 1.19658510e+08,
#        1.19661567e+08, 1.19664624e+08, 1.19667681e+08, 1.19670737e+08,
#        1.19673794e+08, 1.19676851e+08, 1.19679908e+08, 1.19682965e+08,
#        1.19686022e+08, 1.19689079e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119494357.0
0  15 values of dummy. 15 total shots. 45 total images expected.
Run ID: 70778
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.21, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.21 pi, x-center = 994, y-center = 820

 Run ID: 70778

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.21, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.21 pi, x-center = 994, y-center = 820

shot 1/15 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.21, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.21 pi, x-center = 994, y-center = 820

shot 2/15 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.21, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.21 pi, x

In [19]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, compression_point, detuning):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu

        class lightshift(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.ABSORPTION)

                self.xvar('with_imaging', [0,1])
                self.xvar('relative_phase', np.linspace(0., 2*np.pi, 12))

                self.p.v_pd_hf_tweezer_squeeze_power = {compression_point}
                self.p.t_tweezer_squeezer_ramp_2 = 29.e-3

                self.p.frequency_detuned_hf_midpoint = {detuning}
                
                self.p.t_ramsey = 5.e-6
                self.p.t_raman_pulse = self.p.t_raman_pi_pulse / 2

                self.p.amp_imaging = .3
                self.p.t_tweezer_hold = 15.e-3
                self.p.t_tof = 80.e-6
                self.p.t_mot_load = 1.
                self.p.N_repeats = 10

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):

                self.set_imaging_detuning(frequency_detuned=self.p.frequency_detuned_hf_midpoint)
                # self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(ramp_down_painting=True,squeeze=True)
                self.prep_raman()

                self.raman.set(t_phase_origin_mu=now_mu())

                self.raman.pulse(self.p.t_raman_pulse)

                if self.p.with_imaging:
                    self.imaging.on()
                    delay(self.p.t_ramsey)
                    self.imaging.off()
                else:
                    delay(self.p.t_ramsey)

                self.raman.set(relative_phase=self.p.relative_phase)
                self.raman.pulse(self.p.t_raman_pulse)

                self.ttl.raman_shutter.off()

                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

            @kernel
            def run(self):
                self.init_kernel(setup_slm=True)
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                
            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                self.end(expt_filepath)
                """)
        return script

In [20]:
eBuilder = ExptBuilder()

In [21]:
compression_point = np.linspace(.16,3.94,10)

detuning = np.array([-5.15800170e+08, -5.12825258e+08, -5.09850347e+08, -5.06875435e+08,
       -5.03900524e+08, -5.00925612e+08, -4.97950701e+08, -4.94975789e+08,
       -4.92000878e+08, -4.89025967e+08])

for f in range(len(compression_point)):
    print('compression_point detuning = ', compression_point[f], detuning[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(compression_point=compression_point[f],detuning = detuning[f]))
    eBuilder.run_expt()

compression_point detuning =  0.16 -515800170.0
0  20 values of with_imaging. 12 values of relative_phase. 240 total shots. 720 total images expected.
Run ID: 70828
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.0, 'dimension': 0, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 0 um, phase = 0.0 pi, x-center = 994, y-center = 820

 Run ID: 70828
shot 1/240 done
shot 2/240 done
shot 3/240 done
shot 4/240 done
shot 5/240 done
shot 6/240 done
shot 7/240 done
shot 8/240 done
shot 9/240 done
shot 10/240 done
shot 11/240 done
shot 12/240 done
shot 13/240 done
shot 14/240 done
shot 15/240 done
shot 16/240 done
shot 17/240 done
shot 18/240 done
shot 19/240 done
shot 20/240 done
shot 21/240 done
shot 22/240 done
shot 23/240 done
shot 24/240 done
shot 25/240 done
shot 26/240 done
shot 27/240 done
shot 28/240 done
shot 29/240 done
shot 30/240 done
shot 31/240 done
shot 32/240 done
shot 33/240 done
shot 34/240

In [22]:
from kexp import EthernetRelay
from waxx.util.guis.als.als_gui_client import ALSGuiClient
from waxx.util.guis.precilaser.precilaser_gui_client import PrecilaserGuiClient

relay = EthernetRelay()
relay.source_off()

als = ALSGuiClient()
ok = als.run_shutdown_sequence()

precilaser = PrecilaserGuiClient()
ok = precilaser.run_shutdown_sequence()